In [1]:
"""docstring"""
# IPython magig  tools
%load_ext autoreload
%autoreload 2
 
import numpy as np
import pandas as pd

import os
from datetime import datetime, date
from pathlib import Path
from xml.dom import minidom

import aind_data_schema.core.data_description as data_description
import aind_data_schema.components.devices as devices
import aind_data_schema.components.coordinates as coordinates
import aind_data_schema.core.instrument as instrument
import aind_data_schema.core.acquisition as acquisition
import aind_data_schema.components.connections as connection
import aind_data_schema.components.stimulus as stimulus
from aind_data_schema.components.configs import EphysAssemblyConfig, ProbeConfig, MISModuleConfig, ManipulatorConfig
from aind_data_schema.components.identifiers import Person
from aind_data_schema.components.measurements import Calibration
from aind_data_schema_models.brain_atlas import CCFv3


from aind_data_schema_models.organizations import Organization
from aind_data_schema_models.harp_types import HarpDeviceType
from aind_data_schema_models.modalities import Modality
from aind_data_schema_models.stimulus_modality import StimulusModality


import aind_data_schema.components.stimulus as stimulus



from metadata_utils import *
from datetime import datetime, timedelta

from rotations import euler_deg_from_R, R_from_tilt_rx_ry_deg, tilt_rx_ry_deg_from_R, R_from_euler_deg, T_from_R_t


In [2]:
# Organization.from_abbreviation(rig_constants["Basestation Manufacturer"][0])


# Organization.from_abbreviation(rig_constants["Camera Manufacturer"][0]),
# Organization.from_name(rig_constants["Manipulator Manufacturer"][0]),

# Organization.from_abbreviation(rig_constants["Mouse Platform Manufacturer"][0]),
# Organization.from_name(rig_constants["Monitor Manufacturer"][0]),
# Organization.from_name(rig_constants["NIPC Manufacturer"][0]),
# Organization.from_name(rig_constants["NIPX Manufacturer"][0]),


In [3]:
# OBJECT CONSTRUCTION

def build_instrument_object(
    constants_file,
    xl_data,
    insertion_numbers_list,
):
    """
    Returns an ephys_rig from given data.
    
    Parameters
    ----------
    constants_file : str
                        csv filename
    xl_data : dict
                        pre-parsed excel data from xl_opener(targets_insertions_file),
                        keyed by sheet index (0 = targets, 1 = insertions)
    insertion_numbers_list : list of str or int
                        from setup no. column of csv
    
    Returns
    -------
    EphysRig object from aind_data_schema
    """

    rig_constants = csv_opener(constants_file)
    insertion_tracking = xl_data[1]
    target_tracking = xl_data[0]

    insertions = []
    targets = []
    for i in insertion_numbers_list:
        targets.append(get_insertion_details(target_tracking, i))
        insertions.append(get_insertion_details(insertion_tracking, i))

    config_number = None
    if pd.isna(insertions[0]["Config Number"]):
        config_number = 1
    else:
        config_number = int(insertions[0]["Config Number"])

    my_xml = xml_opener(targets)

    port_dict = {}
    port_sources = {}
    print(my_xml)
    
    for i in range(len(my_xml["recording"].getElementsByTagName("NP_PROBE"))):
        port_dict[i] = devices.ProbePort(
            index=xml_attribute(my_xml["recording"], "NP_PROBE", i, "port"),
            probes=[ "Ephys Assembly " +
                xml_attribute(
                    my_xml["recording"], "NP_PROBE", i, "custom_probe_name"
                )
            ],
        )
        port_sources[i] = ("recording", i)
    
    if my_xml["surface_finding"] is not None:
        for i in range(len(my_xml["surface_finding"].getElementsByTagName("NP_PROBE"))):
            if xml_attribute(my_xml["surface_finding"], "NP_PROBE", i, "custom_probe_name") not in [k.probes[0] for k in port_dict.values()]:
                ind = int(np.max(list(port_dict.keys()))) + 1
                port_dict[ind] = devices.ProbePort(
                    index=xml_attribute(my_xml["surface_finding"], "NP_PROBE", i, "port"),
                    probes=[ "Ephys Assembly " +
                        xml_attribute(
                            my_xml["surface_finding"], "NP_PROBE", i, "custom_probe_name"
                        )
                    ],
                )
                port_sources[ind] = ("surface_finding", i)
    else:
        print("No surface finding  found.")

    basestation = devices.NeuropixelsBasestation(
        name=rig_constants["Basestation Name"][0],
        basestation_firmware_version=xml_attribute(
            my_xml["recording"], "NP_PROBE", 0, "bs_firmware_version"
        ),
        bsc_firmware_version=xml_attribute(
            my_xml["recording"], "NP_PROBE", 0, "bsc_firmware_version"
        ),
        slot=int(xml_attribute(my_xml["recording"], "BASESTATION", 0, "Slot")),
        ports=[i for i in port_dict.values()],
        manufacturer=Organization.from_abbreviation(rig_constants["Basestation Manufacturer"][0]),
        model=xml_attribute(
            my_xml["recording"], "NP_PROBE", 0, "bsc_part_number"
        ),
        serial_number=str(
            int(
                xml_attribute(
                    my_xml["recording"], "NP_PROBE", 0, "bsc_serial_number"
                )
            )
        ),
    )

    nipx = devices.DAQDevice(
        data_interface=rig_constants["NIPX Data Interface"][0],
        name=rig_constants["NIPX Name"][0],
        serial_number=rig_constants["NIPX Serial Number"][0],
        manufacturer=Organization.from_name(rig_constants["NIPX Manufacturer"][0]),
        model=rig_constants["NIPX Model"][0]
    )

    nipc = devices.DAQDevice(
        data_interface=rig_constants["NIPC Data Interface"][0],
        name=rig_constants["NIPC Name"][0],
        serial_number=rig_constants["NIPC Serial Number"][0],
        manufacturer=Organization.from_name(rig_constants["NIPC Manufacturer"][0]),
        model=rig_constants["NIPC Model"][0]
    )

    camera_dict = {}
    for i in range(len(rig_constants["Camera Assembly Name"].values())):
        if not pd.isna(rig_constants["Camera Assembly Name"][i]):
            cam_id = str(rig_constants["Camera Assembly Name"][i])
            camera_dict[cam_id] = devices.Camera(
                serial_number=str(rig_constants["Camera Serial Number"][i]),
                model=rig_constants["Camera Model"][0],
                notes=rig_constants["Camera Notes"][0],
                name=str(int(rig_constants["Camera Name"][i])),
                data_interface=rig_constants["Camera Data Interface"][0],
                manufacturer=Organization.from_abbreviation(rig_constants["Camera Manufacturer"][0]),
                size_unit=rig_constants["Camera Size Unit"][0],
                sensor_format=rig_constants["Camera Format Unit"][0],
                sensor_format_unit="Inches",
                chroma=rig_constants["Camera Chroma"][0],
                recording_software=devices.Software(
                    name=rig_constants["Camera Recording Software"][0],
                    version=rig_constants["Camera Recording Software Version"][
                        0
                    ],
                ),
            )

    stick_lens = devices.Lens(
        manufacturer=Organization.INFINITY_PHOTO_OPTICAL,
        name="Stick Lens"
        )

    microscope_assembly_dict = {}
    for i in camera_dict.keys():
        microscope_assembly_dict[i] = devices.CameraAssembly(
            name=str(int(float(i))),
            target=devices.CameraTarget.BRAIN,
            relative_position=[devices.AnatomicalRelative.SUPERIOR],
            camera=camera_dict[i],
            lens=stick_lens,
        )

    probe_dict = {}
    for i in port_dict.keys():
        prb_id = str(port_dict[i].probes[0])
        if xml_attribute(
                my_xml[port_sources[i][0]], "NP_PROBE", port_sources[i][1], "probe_name"
            ) == "Neuropixels 2.0 - Single Shank":
            probe_model = "Neuropixels 2.0 (Single Shank)"
        elif xml_attribute(
                my_xml[port_sources[i][0]], "NP_PROBE", port_sources[i][1], "probe_name"
            ) == "Neuropixels 2.0 - Multishank":
            probe_model = "Neuropixels 2.0 (Multi Shank)"
            
        probe_dict[prb_id] = devices.EphysProbe(
            name="Probe " +prb_id,
            serial_number=xml_attribute(
                my_xml[port_sources[i][0]], "NP_PROBE", port_sources[i][1], "probe_serial_number"
            ),
            manufacturer=Organization.from_abbreviation(rig_constants["Probe Manufacturer"][0]),
            probe_model=probe_model,
            headstage=devices.Device(
                name='Headstage ' + prb_id,
                model=xml_attribute(
                    my_xml[port_sources[i][0]],
                    "NP_PROBE",
                    port_sources[i][1],
                    "headstage_part_number",
                ),
                serial_number=xml_attribute(
                    my_xml[port_sources[i][0]],
                    "NP_PROBE",
                    port_sources[i][1],
                    "headstage_serial_number",
                ),
            ),
        )

    ephys_assembly_dict = {}
    for i in range(len(probe_dict.keys())):
        ephys_assembly_dict[
            xml_attribute(
                my_xml[port_sources[i][0]], "NP_PROBE", port_sources[i][1], "custom_probe_name"
            )
        ] = devices.EphysAssembly(
            name="Ephys Assembly " + xml_attribute(
                my_xml[port_sources[i][0]], "NP_PROBE", port_sources[i][1], "custom_probe_name"
            ),
            probes=[k for (j, k) in probe_dict.items() if j == xml_attribute(
                    my_xml[port_sources[i][0]], "NP_PROBE", port_sources[i][1], "custom_probe_name"
                )],
            manipulator=devices.Manipulator(
                name="Manipulator " + xml_attribute(
                    my_xml[port_sources[i][0]], "NP_PROBE", port_sources[i][1], "custom_probe_name"
                ),
                serial_number=xml_attribute(
                    my_xml[port_sources[i][0]], "NP_PROBE", port_sources[i][1], "custom_probe_name"
                ),
                model=rig_constants["Manipulator Model"][0],
                manufacturer=Organization.NEW_SCALE_TECHNOLOGIES,
            ),
        )

    white_rabbit = devices.HarpDevice(name="White Rabbit", is_clock_generator=True, harp_device_type=HarpDeviceType.WHITERABBIT)

    # white rabbit --> NI PXI digital input
    neuropixels_sync_connection = connection.Connection(
        source_device = white_rabbit.name,
        source_port ="harp_clock",
        target_device = nipx.name,
        target_port = "0"
    )

    # white rabbit --> Neuropixels PXI digital input
    basestation_harp_sync_connection = connection.Connection(
        source_device = white_rabbit.name,
        source_port ="harp_clock",
        target_device = rig_constants["Basestation Name"][0],
        target_port = rig_constants["BSChnl Channel Name"][0]
    )

    components = [basestation, nipx, nipc, white_rabbit]
    components.extend(list(ephys_assembly_dict.values()))
    components.extend(list(microscope_assembly_dict.values()))
    
    date_val = insertions[0]["Date"]

    if isinstance(date_val, str):
        date_val = pd.to_datetime(date_val)

    modification_date = date_val.strftime('%Y-%m-%d')
    behavior_coordinate_system = coordinates.CoordinateSystem(
        name="origin",
        origin=coordinates.Origin.BREGMA,
        axis_unit=coordinates.SizeUnit.MM,
        axes=[
            coordinates.Axis(name=coordinates.AxisName.X, direction=coordinates.Direction.LR),
            coordinates.Axis(name=coordinates.AxisName.Y, direction=coordinates.Direction.AP),
            coordinates.Axis(name=coordinates.AxisName.Z, direction=coordinates.Direction.IS),
        ],
    )
        
    return instrument.Instrument(
        instrument_id = 'ND_Ephys.5', #str(rig_constants["Rig ID Prefix"][0]),
        modification_date = modification_date,
        components = components, 
        modalities = [Modality.ECEPHYS],
        coordinate_system = behavior_coordinate_system,
        connections = [neuropixels_sync_connection, basestation_harp_sync_connection]
    )

In [ ]:
def build_ephys_acquisition_object(
    constants_file,
    xl_data,
    insertion_numbers_list,
):
    """
    Returns an ephys_session from given data.
    
    Parameters
    ----------
    constants_file : str
                        csv filename
    xl_data : dict
                        pre-parsed excel data from xl_opener(targets_insertions_file),
                        keyed by sheet index (0 = targets, 1 = insertions)
    insertion_numbers_list : list of strings or ints
                        each from setup no. column of csv
    
    Returns
    -------
    EphysSession object from aind_data_schema
    """

    # CSV opening and basic organization

    rig_constants = csv_opener(constants_file)

    target_tracking = xl_data[0]
    insertion_tracking = xl_data[1]
    targets = []
    insertions = []
    for i in insertion_numbers_list:
        targets.append(get_insertion_details(target_tracking, i))
        insertions.append(get_insertion_details(insertion_tracking, i))

    # Setting of constants 
    subject_id = str(int(targets[0]["Mouse"]))
    date = pd.to_datetime(insertions[0]["Date"])
    session_number = (
        None  # 1 means it's the first session of the day, 2 means second, etc.
    )
    if pd.isna(insertions[0]["Config Number"]):
        session_number = 1
    else:
        session_number = int(insertions[0]["Config Number"])

    ### XML opening

    xml_dict = xml_opener(targets)

    num_of_recordings_this_session = 0
    for i in xml_dict.keys():
        if xml_dict[i] is not None:
            num_of_recordings_this_session += 1

    def find_node_folder(targets, key):
        """Return the full path to the matching ecephys/Record Node 101 folder."""
        target_names = {str(t[key]) for t in targets}
        session_folders = os.listdir(BASE_DIR)

        for target in target_names:
            for session in session_folders:
                if target in session:
                    return os.path.join(BASE_DIR, session, RECORD_NODE)

        return None


    # --- Use the helper for both cases ---
    ephys_recording_node_folder = find_node_folder(targets, "OEPh Recording")
    surface_find_node_folder   = find_node_folder(targets, "OEPh Recording") # Chnange if surface is in a different file

    # Building the fields

    # TODO -- figure out microscope config
    #microscope_dict = {}
    #for i in range(len(rig_constants["Camera Assembly Name"].values())):
    #    if not pd.isna(rig_constants["Camera Assembly Name"][i]):
    #        cam_id = str(rig_constants["Camera Assembly Name"][i])
    #        microscope_dict[cam_id] = aises.DomeModule(
    #            assembly_name=cam_id,
    #            arc_angle=rig_constants["Camera Arc Angle"][0],
    #            module_angle=rig_constants["Camera Module Angle"][0],
    #            rotation_angle=None, 
    #            angle_unit=rig_constants["Camera Angle Unit"][0],
    #            notes=rig_constants["Camera Notes"][0],
    #        )

    basestation = rig_constants["Basestation Name"][0]
    
    global_coordinate_system = coordinates.CoordinateSystem(
    name = "BREGMA_RAS",
    origin = coordinates.Origin.BREGMA,
    axis_unit=coordinates.SizeUnit.MM,
    axes = [
        coordinates.Axis(name = coordinates.AxisName.ML, direction=coordinates.Direction.LR),
        coordinates.Axis(name = coordinates.AxisName.AP, direction=coordinates.Direction.PA),
        coordinates.Axis(name = coordinates.AxisName.SI, direction=coordinates.Direction.IS)
    ]
    )
    
    # Streams
    stream_dict = {}
    calibrations = [ ]
    for iStream in xml_dict.keys():
        if xml_dict[iStream] is not None:
            # See which inserts for this stream succeeded, set key as tuple:
            # (probe number, target, index number in targets/insertions list)
            successes = {}
            for iInsertionNumber in range(len(insertions)):
                if not pd.isna(insertions[iInsertionNumber]["Manipulator X"]):
                    successes[
                        (
                            str(
                                int(
                                    targets[iInsertionNumber][
                                        "Manip controller SN"
                                    ]
                                )
                            ),
                            targets[iInsertionNumber]["Target"],
                            iInsertionNumber,
                        )
                    ] = None
            # Link successful insertions from spreadsheets to OpenEphys probes
            xml_successes = {}
            for iProbe in range(
                len(xml_dict[iStream].getElementsByTagName("NP_PROBE"))
            ):
                probe_name = xml_attribute(
                    xml_dict[iStream], "NP_PROBE", iProbe, "custom_probe_name"
                )
                for iSuccesses in successes.keys():
                    if iSuccesses[0] == probe_name:
                        xml_successes[
                            iSuccesses
                        ] = iProbe  
                        # The value associated w/ the successes key is the index into xml_attribute
            
            # Ephys modules for each stream
            probe_coordinate_system = coordinates.CoordinateSystem(
                        name = "PROBE_RUFD",
                        origin=coordinates.Origin.TIP,
                        axis_unit=coordinates.SizeUnit.MM,
                        axes = [
                            coordinates.Axis(name = coordinates.AxisName.X, direction=coordinates.Direction.LR),
                            coordinates.Axis(name = coordinates.AxisName.Y, direction=coordinates.Direction.PA),
                            coordinates.Axis(name = coordinates.AxisName.Z, direction=coordinates.Direction.IS)
                        ]
                    )
            
            """Five transformations define the position and orientation of the probe in the global coordinate space:

            Rotation 1: places the probe's local axes in the default position (left-most shank aligned with SI axis, electrodes facing mouse left).
            Rotation 2: probe spin (Z-rotation only). 0 degrees = electrodes facing mouse left, 90 degrees = electrodes facing posterior, 180 degrees = electrodes facing mouse right, 270 degrees = electrodes facing anterior.
            Translation 1: manipulator coordinates in mm. The fourth value should always be 1 to create a valid transformation matrix.
            Rotation 3: Euler angles in degrees defining the probe's orientation.
            Translation 2: measured offset in mm required to place probe tip in the global position. The fourth (depth) coordinate tracks the distance the probe tip advanced along the insertion axis from the brain surface, and is independent of the global coordinate system.

            To find the global position of the probe tip, create a 4 x 4 transformation matrix from Rotation 3 and Translation 2, and apply it to Translation 1.
            """
                
            module_dict = {}
            for iInsertion in xml_successes.keys():
                print(iInsertion)
                probe_name = iInsertion[0]

                # TODO -- fix calibrations
                #calibrations.extend(Calibration(
                #    calibration_date = #
                #    description = #
                #    input = #
                #    output = #
                #    targets[iInsertion[2]]["Manipulator Calibration"]))
                
                arc_angle = targets[iInsertion[2]]["Arc AP"]
                module_angle = targets[iInsertion[2]]["Arc ML"]
                probe_tilt = arc_angle + 14 # actual 0 degrees accounts for mouse head angle (14 degrees forward)
                spin_angle = targets[iInsertion[2]]["Spin"]
                probe_depth = (insertions[iInsertion[2]]["Final Z"] - insertions[iInsertion[2]]["Surface Z"]) / 1000

                manipulator_coords_at_bregma = (targets[iInsertion[2]]["Bregma X"] / 1000,
                                                targets[iInsertion[2]]["Bregma Y"] / 1000,
                                                targets[iInsertion[2]]["Bregma Z"] / 1000)
                
                manipulator_coords_during_exp = [insertions[iInsertion[2]]["Manipulator X"] / 1000,
                                                insertions[iInsertion[2]]["Manipulator Y"] / 1000,
                                                insertions[iInsertion[2]]["Final Z"] / 1000]

                manipulator_offset = np.array(manipulator_coords_during_exp) - np.array(manipulator_coords_at_bregma)

                module_dict[probe_name] = EphysAssemblyConfig(

                    device_name = "Ephys Assembly " + probe_name,

                    manipulator = ManipulatorConfig(
                        device_name = "Manipulator " + probe_name, 
                        coordinate_system=coordinates.CoordinateSystem(
                            name="MIS_MANIP_XYZ",
                            origin=coordinates.Origin.ORIGIN,
                            axis_unit=coordinates.SizeUnit.MM,
                            axes=[
                                coordinates.Axis(name=coordinates.AxisName.X, direction=coordinates.Direction.POS),
                                coordinates.Axis(name=coordinates.AxisName.Y, direction=coordinates.Direction.POS),
                                coordinates.Axis(name=coordinates.AxisName.Z, direction=coordinates.Direction.POS),
                            ],
                        ),
                        local_axis_positions = coordinates.Translation(
                                translation=[insertions[iInsertion[2]]["Manipulator X"]/ 1000,
                                insertions[iInsertion[2]]["Manipulator Y"]/ 1000,
                                insertions[iInsertion[2]]["Final Z"]/ 1000]
                        )
                    ),

                    probes = [ProbeConfig(
                        device_name = probe_name,
                        notes = insertions[iInsertion[2]]["comments"],
                        dye = insertions[iInsertion[2]]["Dye"],
                        primary_targeted_structure = CCFv3.by_acronym(iInsertion[1]),
                        coordinate_system=probe_coordinate_system,
                        transform = [
                            # The following transforms move the probe coordinates into the global coordinate system:
                            # First rotation sets default position, same for every experiment
                            coordinates.Rotation(angles = [90.0, 0.0, -90.0], 
                                                    angles_unit=coordinates.AngleUnit.DEG,
                                                    reference_coordinate_system=coordinates.ReferenceCoordinateSystem.GLOBAL), 

                            # Second rotation is for the arc angle
                            coordinates.Rotation(angles = [probe_tilt, 0, 0], 
                                                    angles_unit=coordinates.AngleUnit.DEG, # Euler angles in degrees
                                                    reference_coordinate_system=coordinates.ReferenceCoordinateSystem.GLOBAL),

                            # Third rotation is for the module angle
                            coordinates.Rotation(angles = [module_angle, 0, 0], angles_unit=coordinates.AngleUnit.DEG, 
                                                                                reference_coordinate_system=coordinates.ReferenceCoordinateSystem.LOCAL,
                                                                                rotation_direction=coordinates.RotationDirection.LEFT_HAND),

                            # Fourth rotation is for spinning the probe along its insertion axis
                            coordinates.Rotation(angles = [0, spin_angle, 0], angles_unit=coordinates.AngleUnit.DEG, 
                                                    reference_coordinate_system=coordinates.ReferenceCoordinateSystem.LOCAL), 

                            # First translation is for the location of the probe at the brain surface
                            coordinates.Translation(translation=[manipulator_offset[0], probe_depth - manipulator_offset[2], manipulator_offset[1]],
                                                    reference_coordinate_system=coordinates.ReferenceCoordinateSystem.LOCAL), 
                                    
                            # Second translation is for the final recording location
                            coordinates.Translation(translation=[0, -probe_depth, 0],
                                                    reference_coordinate_system=coordinates.ReferenceCoordinateSystem.LOCAL), 
                            ]
                    )],
                    
                    modules = [MISModuleConfig(
                        arc_angle = targets[iInsertion[2]]["Arc AP"],
                        module_angle = targets[iInsertion[2]]["Arc ML"],
                        angle_unit = rig_constants["Ephys Module Angle Unit"][0]
                    )]
                )


            if iStream == "recording":
                stream_ts = np.lib.format.open_memmap(
                    str(ephys_recording_node_folder)
                    + "/experiment1/recording1/continuous/Neuropix-PXI-116."
                    + str(list(xml_successes.keys())[0][0])
                    + "/timestamps.npy",
                    mode="r",
                )

  
                xml_date = xml_dict[iStream].getElementsByTagName("DATE")[0].firstChild.nodeValue

                xml_time = ("0" + xml_date[-7:]) if xml_date[-8:][0] == " " else xml_date[-8:]

                # date is already a datetime object
                date_str = date.strftime("%Y-%m-%d")

                # Build start datetime
                start_dt = datetime.strptime(f"{date_str} {xml_time}", "%Y-%m-%d %H:%M:%S")

                # Compute end datetime
                duration_seconds = int(stream_ts[-1] - stream_ts[0])
                end_dt = start_dt + timedelta(seconds=duration_seconds)

                # Convert to ISO8601
                start_iso = start_dt.strftime("%Y-%m-%dT%H:%M:%S.000Z")
                end_iso   = end_dt.strftime("%Y-%m-%dT%H:%M:%S.000Z")

                stream_dict[iStream] = acquisition.DataStream(
                    stream_start_time=start_iso,
                    stream_end_time=end_iso,
                    active_devices=list(module_dict.keys()),
                    configurations=list(module_dict.values()),
                    modalities=[Modality.ECEPHYS],
                    notes = "Recording data stream "        )

            if iStream == "surface_finding":
                stream_ts = np.lib.format.open_memmap(
                    str(surface_find_node_folder)
                    + "/experiment2/recording1/continuous/Neuropix-PXI-116."
                    + str(list(xml_successes.keys())[0][0])
                    + "/timestamps.npy",
                    mode="r",
                )

                # Extract time from XML
                xml_date = xml_dict[iStream].getElementsByTagName("DATE")[0].firstChild.nodeValue
                xml_time = ("0" + xml_date[-7:]) if xml_date[-8:][0] == " " else xml_date[-8:]

                # Convert the already-datetime 'date' into YYYY-MM-DD
                date_str = date.strftime("%Y-%m-%d")

                # Build start datetime
                start_dt = datetime.strptime(f"{date_str} {xml_time}", "%Y-%m-%d %H:%M:%S")

                # Compute end datetime
                duration_seconds = int(stream_ts[-1] - stream_ts[0])
                end_dt = start_dt + timedelta(seconds=duration_seconds)

                # Convert to ISO8601 Zulu timestamps
                start_iso = start_dt.strftime("%Y-%m-%dT%H:%M:%S.000Z")
                end_iso   = end_dt.strftime("%Y-%m-%dT%H:%M:%S.000Z")
                
                stream_dict[iStream] = acquisition.DataStream(
                    stream_start_time=start_iso,
                    stream_end_time=end_iso,
                    active_devices=list(module_dict.keys()),
                    configurations=list(module_dict.values()),
                    modalities=[Modality.ECEPHYS],
                    notes = "Surface finding data stream"
                )

    return acquisition.Acquisition(
        subject_id=subject_id,
        acquisition_start_time=stream_dict["recording"].stream_start_time,
        acquisition_end_time=stream_dict["recording"].stream_end_time,
        # experimenters=[
        #     rig_constants["Experimenters"][i]
        #     for i in range(len(rig_constants["Experimenters"]))
        #     if not pd.isna(rig_constants["Experimenters"][i])
        # ],
        experimenters=['katrina.nguyen', 'tiffany.ona', "yoni.browning"],
        coordinate_system=global_coordinate_system,

        acquisition_type= "AindVrForaging", # Needs to match the acqusition behavior one. 
        calibrations=calibrations,

        ethics_review_id=[str(2414)], # We could automate this
        instrument_id='ND_Ephys.5', #str(rig_constants["Rig ID Prefix"][0]),
        data_streams=list(stream_dict.values())
    )

In [5]:
def build_data_description_object(
    constants_file,
    targets_insertions_file,
    insertion_number,
):
    
    """
    Returns a data_description object from given data.
    
    Parameters
    ----------
    constants_file : str
                        csv filename
    targets_insertions_file : str
                        excel filename containing targets on first sheet and insertions on second
    insertion_number : str or int
                        from setup no. column of csv
    
    Returns
    -------
    DataDescription object from aind_data_schema
    """

    constants = csv_opener(constants_file)

    target_tracking = xl_opener(targets_insertions_file)[0]
    target = get_insertion_details(target_tracking, insertion_number)

    insertion_tracking = xl_opener(targets_insertions_file)[1]
    insertion = get_insertion_details(insertion_tracking, insertion_number)

    date = pd.to_datetime(insertion["Date"])
    return data_description.DataDescription(
        modalities=[Modality.ECEPHYS, Modality.BEHAVIOR],
        license=constants["License"][0],
        subject_id=str(target["Mouse"]),
        creation_time=date.strftime("%Y-%m-%d") + "T" + target["OEPh Recording"][-8:].replace("-", ":"),
        institution=Organization.AIND,
        investigators=[
            Person(name=constants["Experimenters"][i])
            for i in constants["Experimenters"].keys()
            if not pd.isna(constants["Experimenters"][i])
        ],
        funding_source=[data_description.Funding(funder=Organization.AI)],
        data_level=constants["Data Level"][0],
        group=constants["Data Group"][0],
        project_name=constants["Project Name"][0],
        data_summary=constants["Data Summary"][0],
    )

In [6]:
# MASTER WRITE

def write_all_jsons(
    constants_file,
    targets_insertions_file,
    jobs_list,
    save_folder
):

    """
    For each job, writes three JSON files to directory
    
    Parameters
    ----------
    constants_file : str
                        csv filename
    targets_insertions_file : str
                        excel filename containing targets on first sheet and insertions on second
    jobs_list : list of lists of strings or ints
                        from setup no. column of csv
    save_folder : full path string, no slash at the end
                        should lead to an existing folder where new folder and subfolders will be made
    
    Returns
    -------
    None

    Notes
    -----
    Within specified folder, creates a "Metadata JSONs" folder containing folders named after each job/session, each with
    a session, rig, and data description file
    """
    json_folder = Path(str(save_folder) + "/Metadata JSONs")
    json_folder.mkdir(parents=True, exist_ok=True)

    # Parse the tracking spreadsheet once and reuse it for every job, instead of
    # having each builder re-read the whole workbook from disk.
    xl_data = xl_opener(targets_insertions_file)

    for job in jobs_list:
        
        rig_x = build_instrument_object(constants_file, xl_data, job)
        ses_x = build_ephys_acquisition_object(constants_file, xl_data, job)

        formatted = ses_x.subject_id + "_" + ses_x.acquisition_start_time.strftime("%Y-%m-%d_%H-%M-%S")
        print(formatted)
        
        save_path = Path(save_folder).joinpath(formatted)
        # data_folder.mkdir(parents=True, exist_ok=True)

        ses_x.write_standard_file(save_path, suffix="_ecephys.json")
        rig_x.write_standard_file(save_path, suffix="_ecephys.json")
        
        # dd_x = build_data_description_object(constants_file, targets_insertions_file, job[0])
        # dd_x.write_standard_file(data_folder)
    
    print("hooray")
    

In [7]:
rig_constants_csv = r"Z:\scratch\tiffany.ona/ephys_rig_constants_rig5_v2.csv"
tracking_spreadsheet_path = r"Z:\scratch\tiffany.ona\InsertionTracking_VR.xlsx"
insertion_numbers = [[133]]

# Saving the ecephys folder name as a variable to save time later on
BASE_DIR = r"Z:\scratch\tiffany.ona\ephys\sessions"
RECORD_NODE = os.path.join("ecephys", "Record Node 101")

In [8]:
write_all_jsons(rig_constants_csv, tracking_spreadsheet_path, insertion_numbers, BASE_DIR)

Duplicate component names found: ['22438384', '23195700', '23195705', 'Spinview', 'Stick Lens']


Looking for XML files in Z:\scratch\tiffany.ona\ephys\sessions\841314_2026-06-05_11-12-15\ecephys\Record Node 101\settings.xml...
{'recording': <xml.dom.minidom.Document object at 0x00000260D8582510>, 'surface_finding': <xml.dom.minidom.Document object at 0x00000260D85823F0>}
Looking for XML files in Z:\scratch\tiffany.ona\ephys\sessions\841314_2026-06-05_11-12-15\ecephys\Record Node 101\settings.xml...
('46105', 'PL', 0)
('46105', 'PL', 0)
841314_2026-06-05_11-12-15
hooray
